# Multiscale alanine dipeptide in PyTorch

The same molecule at two resolutions, both as **differentiable functions of the coordinates**:

| level | degrees of freedom | energy from |
| --- | --- | --- |
| all-atom | 22 atoms | AMBER14 + GBn2 implicit solvent, via OpenMM |
| coarse-grained | 5 beads `C–N–CA–C–N` | Boltzmann inversion of the all-atom trajectory |

Both obey one contract — the `PotentialLike` protocol of `md_simulations.torch_potentials`:

> positions in **nm**, shape `(N, 3)` or `(F, N, 3)` → energy in **kJ/mol**, shape `()` or `(F,)`,
> differentiable with respect to the positions.

Anything you write against that contract (a loss, a sampler, a learned correction) works at
either resolution. Setup is in [README.md](README.md).

The reference frames are a 100 ns temperature-replica-exchange run at 330 K, subsampled and
relaxed into the unconstrained force field by [prepare_data.py](prepare_data.py) — replica
exchange because plain MD at this temperature almost never visits the $\alpha_L$ basin
(0.0004 % of frames against 1.7 % here).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np
import torch

torch.set_default_dtype(torch.float64)

HERE = Path.cwd()
CONFIG = HERE / "ala2-amber14-implicit-330K.yaml"
DATA = HERE / "data"

TEMPERATURE = 330.0  # K, the trajectory's temperature
KT = 0.00831446261815324 * TEMPERATURE  # kJ/mol

topology = md.load_topology(DATA / "ala2_topology.pdb")
xyz = torch.tensor(np.load(DATA / "ala2_330K_xyz.npy"), dtype=torch.float64)  # (F, 22, 3) nm

print(f"kT = {KT:.3f} kJ/mol | {tuple(xyz.shape)} frames of {topology.n_atoms} atoms")

## 1. The all-atom potential

`build_potential_from_file` reads the YAML that defines the system, builds the OpenMM system
behind it, and hands back something you call with torch tensors.

OpenMM is opaque to autograd, so the bridge is a hand-written `torch.autograd.Function`
(`md_simulations.torch_potentials.bridge`): `forward` asks OpenMM for energy *and* forces,
`backward` returns `-grad_output * forces`. Worth reading — it is 40 lines, and it is the whole
trick. One consequence: **first derivatives only**, no double-backward.

Two details that are easy to get wrong:

- The config sets `constraints: none`. The usual `hbonds` setting makes the twelve X–H bonds
  rigid and **deletes their energy term**, so those coordinates would simply be absent from the
  potential you differentiate. Unconstrained, all 21 bonds are in it — at the price of a 0.5 fs
  timestep if you ever integrate it.
- The positions must be in the **engine's atom order**, which need not be the order of the PDB
  the system was built from, because the force field adds the hydrogens.
  `data/ala2_topology.pdb` and the shipped trajectory are both in that order; the assertion
  below is what catches it when they are not.

In [ ]:
from md_simulations.torch_potentials import build_potential_from_file

aa_potential = build_potential_from_file(topology, CONFIG, data_root=DATA)

x = xyz[0].clone().requires_grad_(True)  # (22, 3) nm
energy = aa_potential(x)  # scalar, kJ/mol
energy.backward()
forces = -x.grad  # (22, 3), kJ/mol/nm  —  F = -dU/dx

# a mis-ordered trajectory is the classic failure here, and it is loud: thousands of
# kJ/mol instead of hundreds, with forces to match
assert abs(energy.item()) < 1e3, "atom order mismatch between the positions and the engine"

print(f"U       = {energy.item():.2f} kJ/mol")
print(f"max |F| = {forces.abs().max():.1f} kJ/mol/nm")
print(f"sum F   = {forces.sum(0).numpy().round(6)}   (translation invariance)")

A leading frame axis works too. It is not a fused batch — OpenMM is single-system, so
`Potential` loops internally — but the calling convention matches the CG model below.

In [ ]:
with torch.no_grad():
    print(aa_potential(xyz[:4]).numpy().round(1), "kJ/mol")

## 2. The coarse-grained map

The five beads are the backbone path carrying the two torsions that define this molecule:

```
φ = C–N–CA–C          ψ = N–CA–C–N
C(ACE) — N(ALA) — CA(ALA) — C(ALA) — N(NME)
   0        1         2        3        4
```

so φ and ψ are bead dihedrals `0-1-2-3` and `1-2-3-4`. Each bead sits on one atom, so the map is
a slice — written here as a `(5, 22)` matrix because that is also how a centre-of-mass map would
look, and as a matmul it stays differentiable.

In [ ]:
bead_indices = topology.select(
    "(resname ACE and name C) or (resname ALA and name N CA C) or (resname NME and name N)"
)
cg_topology = topology.subset(bead_indices)

cg_map = torch.zeros(len(bead_indices), topology.n_atoms)
cg_map[torch.arange(len(bead_indices)), torch.as_tensor(bead_indices.copy())] = 1.0

cg_xyz = cg_map @ xyz  # (F, 5, 3) nm — the CG trajectory

print([str(a) for a in cg_topology.atoms], "->", tuple(cg_xyz.shape))

## 3. Internal coordinates as torch ops

The CG potential is built from bond lengths, angles and dihedrals. They have to be torch
functions so autograd can reach the Cartesian coordinates through them. All three are batched
over a leading frame axis.

In [ ]:
def bond_lengths(r, pairs):
    """(..., N, 3) nm -> (..., n_pairs) nm."""
    return torch.linalg.norm(r[..., pairs[:, 1], :] - r[..., pairs[:, 0], :], dim=-1)


def bond_angles(r, triples):
    """(..., N, 3) nm -> (..., n_triples) rad in [0, pi]."""
    a, b, c = (r[..., triples[:, i], :] for i in range(3))
    u, v = a - b, c - b
    # atan2(|u x v|, u.v), not acos: the derivative stays finite at 0 and pi
    return torch.atan2(torch.linalg.cross(u, v, dim=-1).norm(dim=-1), (u * v).sum(-1))


def dihedrals(r, quads):
    """(..., N, 3) nm -> (..., n_quads) rad in [-pi, pi)."""
    p0, p1, p2, p3 = (r[..., quads[:, i], :] for i in range(4))
    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2  # b0 reversed: the IUPAC sign convention
    b1 = b1 / b1.norm(dim=-1, keepdim=True)
    v = b0 - (b0 * b1).sum(-1, keepdim=True) * b1  # components perpendicular to the
    w = b2 - (b2 * b1).sum(-1, keepdim=True) * b1  # central bond
    return torch.atan2((torch.linalg.cross(b1, v, dim=-1) * w).sum(-1), (v * w).sum(-1))


BONDS = torch.tensor([[0, 1], [1, 2], [2, 3], [3, 4]])
ANGLES = torch.tensor([[0, 1, 2], [1, 2, 3], [2, 3, 4]])
TORSIONS = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 4]])  # phi, psi

In [ ]:
# cross-check the torsions against mdtraj, on the circle so a wrap at +-pi does not count
traj = md.Trajectory(xyz.numpy().astype(np.float32), topology)
reference = np.stack([md.compute_phi(traj)[1][:, 0], md.compute_psi(traj)[1][:, 0]], axis=1)
delta = dihedrals(cg_xyz, TORSIONS).numpy() - reference
print("max deviation from mdtraj:", np.abs(np.angle(np.exp(1j * delta))).max())

## 4. Boltzmann inversion

For one internal coordinate $q$ sampled at temperature $T$,

$$U(q) = -k_\mathrm{B}T \,\ln \frac{P(q)}{J(q)}$$

The Jacobian $J$ is the volume the coordinate sweeps — $r^2$ for a bond, $\sin\theta$ for an
angle, $1$ for a dihedral. Dividing it out separates the potential from the coordinate's own
entropy; skip it and the bond minima come out too long.

Two things this *is not*, worth knowing before the curves look convincing:

1. It is a **potential of mean force**. It already contains the averaged effect of everything we
   removed, so it is only valid at 330 K.
2. Summing one-dimensional profiles assumes the coordinates are **independent**. For alanine
   dipeptide φ and ψ are not, so the CG model reproduces each marginal but not their coupling.

In [ ]:
def boltzmann_invert(samples, grid, jacobian=None):
    """Histogram each coordinate and invert it into a free-energy profile.

    :param samples: (F, n_terms) values of each coordinate over the trajectory.
    :param grid: (n_bins,) uniform bin centres, shared by the terms.
    :param jacobian: (n_bins,) phase-space volume per bin, or None for 1.
    :return: (n_terms, n_bins) energies in kJ/mol, each shifted to a minimum of 0.
    """
    half = 0.5 * float(grid[1] - grid[0])
    counts = torch.stack(
        [
            torch.histc(q, bins=len(grid), min=float(grid[0]) - half, max=float(grid[-1]) + half)
            for q in samples.T
        ]
    )
    density = counts / counts.sum(-1, keepdim=True)
    if jacobian is not None:
        density = density / jacobian

    # an empty bin means "never visited", not "infinitely high": cap it at a finite
    # wall so the potential and its gradient stay usable
    floor = density[density > 0].min() * 1e-3
    energy = -KT * torch.log(density.clamp_min(floor))
    return energy - energy.min(dim=-1, keepdim=True).values

In [ ]:
# each group gets the resolution its distribution deserves: bonds are narrow, torsions
# are broad but sparsely sampled across their barriers
bond_grid = torch.linspace(0.115, 0.175, 80)  # nm
angle_grid = torch.linspace(1.3, 2.6, 60)  # rad
torsion_grid = torch.linspace(-np.pi, np.pi, 37)[:-1]  # rad, periodic (10 deg bins)

bond_energies = boltzmann_invert(bond_lengths(cg_xyz, BONDS), bond_grid, bond_grid**2)
angle_energies = boltzmann_invert(bond_angles(cg_xyz, ANGLES), angle_grid, torch.sin(angle_grid))
torsion_energies = boltzmann_invert(dihedrals(cg_xyz, TORSIONS), torsion_grid)

print(bond_energies.shape, angle_energies.shape, torsion_energies.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, grid, energies, xlabel, degrees in [
    (axes[0], bond_grid, bond_energies, "bond length / nm", False),
    (axes[1], angle_grid, angle_energies, "angle / deg", True),
    (axes[2], torsion_grid, torsion_energies, "torsion / deg", True),
]:
    q = np.degrees(grid.numpy()) if degrees else grid.numpy()
    ax.plot(q, energies.T.numpy())
    ax.set(xlabel=xlabel, ylim=(0, 30))
axes[0].set_ylabel("U / kJ mol$^{-1}$")
axes[2].legend(["$\\phi$", "$\\psi$"])
fig.tight_layout()

The bond and angle profiles come out close to parabolas — those two could be replaced by a
harmonic fit with two parameters each. The torsions are what need the full numerical profile:
`φ` has the two basins that make this molecule interesting.

## 5. The CG potential

A table is numbers on a grid; a potential has to be a differentiable function. Linear
interpolation between grid points is enough — differentiable in $q$, and a dozen lines.

In [ ]:
def interpolate(q, grid, energies, periodic=False):
    """Linearly interpolate tabulated energies at the coordinate values q.

    :param q: (..., n_terms) coordinate values.
    :param grid: (n_bins,) uniform bin centres.
    :param energies: (n_terms, n_bins) tabulated energies in kJ/mol.
    :param periodic: wrap past the last bin onto the first (torsions) instead of clamping.
    :return: (...,) summed energy of all terms, in kJ/mol.
    """
    n_bins = len(grid)
    position = (q - grid[0]) / (grid[1] - grid[0])  # continuous bin index
    position = position % n_bins if periodic else position.clamp(0, n_bins - 1 - 1e-9)

    low = position.floor().long()
    high = (low + 1) % n_bins if periodic else (low + 1).clamp(max=n_bins - 1)
    weight = position - low  # carries the gradient w.r.t. q

    table = energies.expand(*q.shape[:-1], *energies.shape)
    below = table.gather(-1, low.unsqueeze(-1)).squeeze(-1)
    above = table.gather(-1, high.unsqueeze(-1)).squeeze(-1)
    return ((1 - weight) * below + weight * above).sum(-1)

In [ ]:
class CGPotential(torch.nn.Module):
    """Separable CG potential: tabulated bond, angle and torsion profiles.

    Satisfies PotentialLike, so it is interchangeable with the all-atom potential.
    """

    def __init__(self, topology):
        super().__init__()
        self._topology = topology

    @property
    def n_atoms(self):
        """Number of CG beads."""
        return self._topology.n_atoms

    @property
    def topology(self):
        """The bound coarse-grained mdtraj Topology."""
        return self._topology

    def forward(self, positions, unitcell_lengths=None, unitcell_angles=None):
        """:param positions: (5, 3) or (F, 5, 3) bead positions in nm.
        :param unitcell_lengths: ignored, this system is non-periodic.
        :param unitcell_angles: ignored.
        :return: () or (F,) energy in kJ/mol, differentiable w.r.t. positions.
        """
        return (
            interpolate(bond_lengths(positions, BONDS), bond_grid, bond_energies)
            + interpolate(bond_angles(positions, ANGLES), angle_grid, angle_energies)
            + interpolate(
                dihedrals(positions, TORSIONS), torsion_grid, torsion_energies, periodic=True
            )
        )


from md_simulations.torch_potentials import PotentialLike

cg_potential = CGPotential(cg_topology)
print("satisfies PotentialLike:", isinstance(cg_potential, PotentialLike))

### The same four lines as section 1, at the other resolution

In [ ]:
b = cg_xyz[0].clone().requires_grad_(True)  # (5, 3) nm

cg_energy = cg_potential(b)
cg_energy.backward()
cg_forces = -b.grad

print(f"U_CG    = {cg_energy.item():.2f} kJ/mol")
print(f"max |F| = {cg_forces.abs().max():.1f} kJ/mol/nm")